# Fase 6 — Deploy
Entrega 1: plano de metas por loja, calculado pela posição da loja dentro do próprio cluster.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

SEED = 42
RAIZ = Path('..')
PROCESSADOS = RAIZ / 'dados' / 'processados'
ESTADO = RAIZ / 'estado'
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 250)

base = pd.read_parquet(PROCESSADOS / 'base_lojas_clusters.parquet')
perfil_lojas = pd.read_pickle(ESTADO / 'perfil_lojas.pkl')

## 1. Parâmetros do plano de metas

In [2]:
NOMES_CLUSTER = {0: 'Bairro competitivo de cesta pequena', 1: 'Destino de cesta grande', 2: 'Sazonal',
                 3: 'Dependente de promoção', 4: 'Alto fluxo, cesta pequena'}
META_MANUTENCAO = 0.02   # meta de quem já está na mediana do cluster ou acima
CAPTURA_GAP = 0.25       # parcela do gap até a mediana que a loja atrasada recebe como meta adicional
LIMIAR_ACAO = 0.10       # gap a partir do qual a loja é marcada como "Ação"
META_REDE = 0.05         # compromisso assinado com o conselho, ponderado pelo faturamento

## 2. Metas iniciais (regra de ajuste relativo)

In [3]:
plano = pd.DataFrame(index=base.index)
plano['cluster'] = base['cluster']
plano['nome_cluster'] = base['cluster'].map(NOMES_CLUSTER)
plano['venda_media_dia'] = base['venda_media_dia']
plano['benchmark_grupo'] = base.groupby('cluster')['venda_media_dia'].transform('median')
# gap = quanto a loja precisa crescer para chegar à mediana dos pares; zero para quem já está na mediana ou acima
plano['gap_pct'] = ((plano['benchmark_grupo'] - plano['venda_media_dia']) / plano['venda_media_dia']).clip(lower=0)
plano['meta_inicial'] = META_MANUTENCAO + CAPTURA_GAP * plano['gap_pct']
plano['prioridade'] = np.where(plano['gap_pct'] >= LIMIAR_ACAO, 'Ação', 'Acompanhamento')
plano['dias_operacao_ano'] = perfil_lojas.loc[plano.index, 'pct_dias_abertos'] * 365
plano['faturamento_anual'] = plano['venda_media_dia'] * plano['dias_operacao_ano']
print(plano[['venda_media_dia', 'benchmark_grupo', 'gap_pct', 'meta_inicial']].describe().round(4).to_string())

       venda_media_dia  benchmark_grupo    gap_pct  meta_inicial
count        1115.0000        1115.0000  1115.0000     1115.0000
mean         6934.6320        6680.2584     0.1258        0.0515
std          2383.9825        1118.8744     0.1933        0.0483
min          2703.7366        5476.3703     0.0000        0.0200
25%          5322.3000        5476.3703     0.0000        0.0200
50%          6589.9485        6959.9244     0.0000        0.0200
75%          7964.2006        6959.9244     0.2013        0.0703
max         21757.4834        8837.1051     1.1856        0.3164


## 3. Checagem contra o compromisso de +5%

In [4]:
def crescimento_ponderado(metas):
    return (plano['faturamento_anual'] * metas).sum() / plano['faturamento_anual'].sum()

faturamento_rede = plano['faturamento_anual'].sum()
media_simples = plano['meta_inicial'].mean()
ponderado = crescimento_ponderado(plano['meta_inicial'])
print(f'média simples das metas iniciais : {media_simples:.2%}')
print(f'crescimento ponderado da rede    : {ponderado:.2%}  (o que de fato soma em R$)')
print(f'distância do compromisso de 5%   : {(ponderado - META_REDE) * 100:+.2f} p.p. = R$ {(ponderado - META_REDE) * faturamento_rede / 1e6:+,.1f} mi por ano')
print(f'faturamento anual da rede        : R$ {faturamento_rede / 1e6:,.1f} mi')

quem_puxa = plano.groupby(plano['gap_pct'] > 0).agg(lojas=('meta_inicial', 'size'), meta_media=('meta_inicial', 'mean'),
                                                     participacao_faturamento=('faturamento_anual', lambda s: s.sum() / faturamento_rede))
quem_puxa.index = ['na mediana ou acima', 'abaixo da mediana']
print('\n', quem_puxa.round(4).to_string())

média simples das metas iniciais : 5.15%
crescimento ponderado da rede    : 4.19%  (o que de fato soma em R$)
distância do compromisso de 5%   : -0.81 p.p. = R$ -19.0 mi por ano
faturamento anual da rede        : R$ 2,345.7 mi

                      lojas  meta_media  participacao_faturamento
na mediana ou acima    559      0.0200                    0.6088
abaixo da mediana      556      0.0831                    0.3912


## 4. Calibração: um fator único sobre todas as metas, preservando a ordem relativa

In [5]:
fator_calibracao = META_REDE / ponderado
plano['meta_pct'] = plano['meta_inicial'] * fator_calibracao
print(f'fator de calibração: {fator_calibracao:.4f}')
print(f'meta de manutenção após calibração: {META_MANUTENCAO * fator_calibracao:.2%}')
print(f'média simples final   : {plano["meta_pct"].mean():.2%}')
print(f'crescimento ponderado : {crescimento_ponderado(plano["meta_pct"]):.4%}')
print(f'ganho anual previsto  : R$ {(plano["faturamento_anual"] * plano["meta_pct"]).sum() / 1e6:,.1f} mi')

fator de calibração: 1.1932
meta de manutenção após calibração: 2.39%
média simples final   : 6.14%
crescimento ponderado : 5.0000%
ganho anual previsto  : R$ 117.3 mi


## 5. Distribuição das metas

In [6]:
print(plano['meta_pct'].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99]).round(4).to_string())
faixas = pd.cut(plano['meta_pct'], [0, 0.0239, 0.05, 0.10, 0.15, 0.20, 1],
                labels=['manutenção (2,4%)', '2,4% a 5%', '5% a 10%', '10% a 15%', '15% a 20%', 'acima de 20%'])
print('\n', faixas.value_counts().sort_index().to_frame('lojas').assign(pct=lambda d: (d['lojas'] / len(plano) * 100).round(1)).to_string())

count    1115.0000
mean        0.0614
std         0.0577
min         0.0239
10%         0.0239
25%         0.0239
50%         0.0239
75%         0.0839
90%         0.1417
99%         0.2642
max         0.3775

                    lojas   pct
meta_pct                      
manutenção (2,4%)    559  50.1
2,4% a 5%            121  10.9
5% a 10%             222  19.9
10% a 15%            115  10.3
15% a 20%             52   4.7
acima de 20%          46   4.1


In [7]:
resumo_cluster = plano.groupby(['cluster', 'nome_cluster']).agg(
    lojas=('meta_pct', 'size'), benchmark=('benchmark_grupo', 'first'),
    lojas_acao=('prioridade', lambda s: (s == 'Ação').sum()),
    meta_media=('meta_pct', 'mean'), meta_maxima=('meta_pct', 'max'),
    faturamento_anual_mi=('faturamento_anual', lambda s: s.sum() / 1e6))
resumo_cluster['crescimento_ponderado'] = [crescimento_ponderado(plano['meta_pct'].where(plano['cluster'] == c, 0)) * faturamento_rede
                                           / plano.loc[plano['cluster'] == c, 'faturamento_anual'].sum() for c in resumo_cluster.index.get_level_values(0)]
print(resumo_cluster.round(4).to_string())
print('\n', plano.groupby('prioridade').agg(lojas=('meta_pct', 'size'), meta_media=('meta_pct', 'mean'), meta_min=('meta_pct', 'min'), meta_max=('meta_pct', 'max')).round(4).to_string())

                                             lojas  benchmark  lojas_acao  meta_media  meta_maxima  faturamento_anual_mi  crescimento_ponderado
cluster nome_cluster                                                                                                                           
0       Bairro competitivo de cesta pequena    292  5476.3703         106      0.0589       0.2819              504.9846                 0.0485
1       Destino de cesta grande                414  6959.9244         152      0.0582       0.2856              884.5714                 0.0489
2       Sazonal                                 65  6324.8010          25      0.0746       0.2657              126.3191                 0.0582
3       Dependente de promoção                 165  5909.2449          64      0.0668       0.3775              305.6598                 0.0519
4       Alto fluxo, cesta pequena              179  8837.1051          70      0.0631       0.3067              524.1343                

In [8]:
# sensibilidade do limiar de prioridade
for limiar in [0.0001, 0.05, 0.10, 0.20]:
    print(f'gap >= {limiar:.0%}: {(plano["gap_pct"] >= limiar).sum()} lojas em Ação')
print('\nlojas com meta acima de 20%:'); print(plano[plano['meta_pct'] > 0.20][['cluster', 'venda_media_dia', 'benchmark_grupo', 'gap_pct', 'meta_pct']].round(3).to_string())

gap >= 0%: 556 lojas em Ação
gap >= 5%: 487 lojas em Ação
gap >= 10%: 417 lojas em Ação
gap >= 20%: 282 lojas em Ação

lojas com meta acima de 20%:
      cluster  venda_media_dia  benchmark_grupo  gap_pct  meta_pct
loja                                                              
50          1         4256.071         6959.924    0.635     0.213
83          1         3887.706         6959.924    0.790     0.260
130         4         5121.063         8837.105    0.726     0.240
198         3         2900.605         5909.245    1.037     0.333
208         0         2936.291         5476.370    0.865     0.282
210         0         3193.985         5476.370    0.715     0.237
219         0         3133.712         5476.370    0.748     0.247
232         3         3376.759         5909.245    0.750     0.248
237         4         4535.991         8837.105    0.948     0.307
254         0         3005.983         5476.370    0.822     0.269
263         1         3707.516         6959.924 

In [9]:
plano.index.name = 'loja'
plano.to_pickle(ESTADO / 'plano_metas.pkl')
plano.reset_index()[['loja', 'cluster', 'nome_cluster', 'venda_media_dia', 'dias_operacao_ano']].to_csv(PROCESSADOS / 'insumo_plano_metas.csv', index=False)

## 6. Teto de meta e recalibração
A regra sem teto gerou 46 lojas com meta acima de 20% (máximo de 37,8%). Meta desse tamanho repete o problema do case:
o gerente desiste no primeiro trimestre. As metas passam a ter um teto, e o fator de calibração é recalculado
para que a rede continue somando exatamente +5% ponderado pelo faturamento.

In [2]:
TETO_META = 0.20

def calibrar_com_teto(meta_inicial, pesos, alvo, teto, max_iter=50):
    # o fator sobe para compensar o que o teto corta; repete até o conjunto de lojas no teto parar de mudar
    fator = alvo / ((pesos * meta_inicial).sum() / pesos.sum())
    for _ in range(max_iter):
        no_teto = meta_inicial * fator > teto
        novo = (alvo * pesos.sum() - teto * pesos[no_teto].sum()) / (pesos[~no_teto] * meta_inicial[~no_teto]).sum()
        if np.isclose(novo, fator, rtol=0, atol=1e-12):
            break
        fator = novo
    return fator

fator_sem_teto = META_REDE / crescimento_ponderado(plano['meta_inicial'])
fator_com_teto = calibrar_com_teto(plano['meta_inicial'], plano['faturamento_anual'], META_REDE, TETO_META)
plano['meta_sem_teto'] = plano['meta_inicial'] * fator_sem_teto
plano['meta_pct'] = (plano['meta_inicial'] * fator_com_teto).clip(upper=TETO_META)
plano['no_teto'] = plano['meta_inicial'] * fator_com_teto > TETO_META

print(f'fator sem teto: {fator_sem_teto:.4f} | fator com teto: {fator_com_teto:.4f}')
print(f'meta de manutenção: {META_MANUTENCAO * fator_sem_teto:.2%} -> {META_MANUTENCAO * fator_com_teto:.2%}')
print(f'lojas no teto de {TETO_META:.0%}: {plano["no_teto"].sum()}')
print(f'crescimento ponderado: {crescimento_ponderado(plano["meta_pct"]):.4%} | média simples: {plano["meta_pct"].mean():.2%}')
print(f'ganho anual previsto: R$ {(plano["faturamento_anual"] * plano["meta_pct"]).sum() / 1e6:,.1f} mi')

# o que o teto tira das lojas atrasadas vai para quem está na mediana ou acima
comparacao_teto = pd.DataFrame({
    'sem_teto': plano.groupby(plano['gap_pct'] > 0)['meta_sem_teto'].mean(),
    'com_teto': plano.groupby(plano['gap_pct'] > 0)['meta_pct'].mean()})
comparacao_teto.index = ['na mediana ou acima', 'abaixo da mediana']
print('\nmeta média:'); print(comparacao_teto.round(4).to_string())

fator sem teto: 1.1932 | fator com teto: 1.2202
meta de manutenção: 2.39% -> 2.44%
lojas no teto de 20%: 47
crescimento ponderado: 5.0000% | média simples: 6.05%
ganho anual previsto: R$ 117.3 mi

meta média:
                     sem_teto  com_teto
na mediana ou acima    0.0239    0.0244
abaixo da mediana      0.0991    0.0969


In [3]:
faixas = pd.cut(plano['meta_pct'], [0, META_MANUTENCAO * fator_com_teto + 1e-9, 0.05, 0.10, 0.15, 0.1999999, 1],
                labels=['manutenção', 'manutenção a 5%', '5% a 10%', '10% a 15%', '15% a 20%', 'no teto (20%)'])
print(faixas.value_counts().sort_index().to_frame('lojas').assign(pct=lambda d: (d['lojas'] / len(plano) * 100).round(1)).to_string())
print('\n', plano['meta_pct'].describe(percentiles=[0.25, 0.5, 0.75, 0.9]).round(4).to_string())

                 lojas   pct
meta_pct                    
manutenção         559  50.1
manutenção a 5%    117  10.5
5% a 10%           214  19.2
10% a 15%          121  10.9
15% a 20%           57   5.1
no teto (20%)       47   4.2

 count    1115.0000
mean        0.0605
std         0.0516
min         0.0244
25%         0.0244
50%         0.0244
75%         0.0858
90%         0.1449
max         0.2000


## 7. Marcações recomendadas na avaliação
- **Subgrupo 7 dias:** lojas do cluster 4 que abrem aos domingos (perfil de conveniência, ticket muito abaixo do resto do cluster).
- **Fronteira de cluster:** lojas que ficaram no próprio cluster em menos de 80% das 100 subamostras do teste de estabilidade.

In [4]:
permanencia = pd.read_pickle(ESTADO / 'permanencia_lojas.pkl')
plano['abre_domingo'] = (base.loc[plano.index, 'prop_domingos_abertos'] > 0).astype(int)
plano['permanencia_cluster'] = permanencia.loc[plano.index, 'taxa']
subgrupo = (plano['cluster'] == 4) & (plano['abre_domingo'] == 1)
fronteira = plano['permanencia_cluster'] < 0.80
plano['observacao'] = np.select([subgrupo & fronteira, subgrupo, fronteira],
                                ['Subgrupo 7 dias; fronteira de cluster', 'Subgrupo 7 dias', 'Fronteira de cluster'], default='')
print(plano['observacao'].value_counts().to_string())
c4 = plano['cluster'] == 4
print('\ncluster 4 - ticket médio: subgrupo 7 dias', round(base.loc[plano.index[subgrupo], 'ticket_medio'].mean(), 2),
      '| demais', round(base.loc[plano.index[c4 & ~subgrupo], 'ticket_medio'].mean(), 2))
print('lojas do subgrupo que são tipo b:', int((base.loc[plano.index[subgrupo], 'tipo_loja'] == 'b').sum()), 'de', int(subgrupo.sum()))
print('fronteira por prioridade:', plano.loc[fronteira, 'prioridade'].value_counts().to_dict())

observacao
                        980
Fronteira de cluster    122
Subgrupo 7 dias          13

cluster 4 - ticket médio: subgrupo 7 dias 5.18 | demais 8.27
lojas do subgrupo que são tipo b: 13 de 13
fronteira por prioridade: {'Acompanhamento': 69, 'Ação': 53}


In [5]:
plano.to_pickle(ESTADO / 'plano_metas.pkl')
plano.reset_index()[['loja', 'cluster', 'nome_cluster', 'venda_media_dia', 'dias_operacao_ano', 'abre_domingo', 'permanencia_cluster']] \
    .to_csv(PROCESSADOS / 'insumo_plano_metas.csv', index=False)
plano.reset_index()[['loja', 'cluster', 'nome_cluster', 'venda_media_dia', 'benchmark_grupo', 'gap_pct', 'meta_pct', 'prioridade', 'observacao']] \
    .to_csv(PROCESSADOS / 'plano_metas.csv', index=False)

## 8. Classificação de lojas novas: regras de uso do modelo
Entrega 2: uma loja nova é enquadrada com o escalonador e o K-Means já treinados, sem novo ajuste.
Antes de exportar, três regras de uso precisam de evidência: quanto histórico a loja precisa ter,
quando a classificação está perto da fronteira entre clusters e quando o perfil está fora do que o modelo viu.

In [2]:
import sys
import joblib
sys.path.insert(0, str(RAIZ.resolve()))
from codigo.configuracao import FEATURES
from codigo.indicadores import calcular_indicadores

escalonador = joblib.load(RAIZ / 'modelos' / 'escalonador.joblib')
kmeans = joblib.load(RAIZ / 'modelos' / 'kmeans_k5.joblib')
vendas = pd.read_csv(RAIZ / 'dados' / 'brutos' / 'vendas.csv', dtype={'feriado_estadual': str})
cadastro = pd.read_csv(RAIZ / 'dados' / 'brutos' / 'loja.csv')
MEDIANA_DISTANCIA = cadastro['distancia_concorrente'].median()

def escalonar(tabela):
    return pd.DataFrame(escalonador.transform(np.log1p(tabela[FEATURES])), columns=FEATURES)

def prever(indicadores):
    validas = indicadores.dropna(subset=FEATURES)
    return pd.Series(kmeans.predict(escalonar(validas)), index=validas['loja'])

# o cálculo do pacote de deploy reproduz a base do treino?
recalculado = calcular_indicadores(vendas, cadastro, MEDIANA_DISTANCIA).set_index('loja')
print('maior diferença absoluta nos indicadores:', f"{(recalculado[FEATURES] - base[FEATURES]).abs().max().max():.1e}")
print('rótulos iguais aos do treino:', int((prever(recalculado.reset_index()) == base['cluster']).sum()), 'de', len(base))

maior diferença absoluta nos indicadores: 1.2e-16
rótulos iguais aos do treino: 1115 de 1115


### 8.1 Histórico mínimo
Para cada loja, recorto só os primeiros N meses de operação (como se fosse nova) e também os últimos N meses,
recalculo os indicadores e comparo o cluster com o do histórico completo.

In [3]:
vendas['data'] = pd.to_datetime(vendas['data'])
dias_operacao = vendas[(vendas['aberta'] == 1) & (vendas['faturamento'] > 0)]
inicio = dias_operacao.groupby('loja')['data'].min()
fim = dias_operacao.groupby('loja')['data'].max()

linhas, por_cluster = [], {}
for meses in [1, 2, 3, 4, 6, 9, 12, 18, 24]:
    for janela, mascara in {
        'primeiros meses': vendas['data'] < vendas['loja'].map(inicio) + pd.DateOffset(months=meses),
        'últimos meses': vendas['data'] > vendas['loja'].map(fim) - pd.DateOffset(months=meses),
    }.items():
        rotulos = prever(calcular_indicadores(vendas[mascara], cadastro, MEDIANA_DISTANCIA))
        iguais = rotulos == base.loc[rotulos.index, 'cluster']
        if meses in (12, 24):
            por_cluster[(meses, janela)] = iguais.groupby(base.loc[rotulos.index, 'cluster']).mean() * 100
        linhas.append({'meses': meses, 'janela': janela, 'lojas_classificadas': len(rotulos),
                       'concordancia_pct': iguais.mean() * 100,
                       'pior_cluster_pct': iguais.groupby(base.loc[rotulos.index, 'cluster']).mean().min() * 100})
historico_minimo = pd.DataFrame(linhas).pivot(index='meses', columns='janela')
print(historico_minimo.round(1).to_string())
print('\nconcordância por cluster (%), 12 e 24 meses:')
print(pd.DataFrame(por_cluster).round(1).to_string())

       lojas_classificadas               concordancia_pct               pior_cluster_pct              
janela     primeiros meses últimos meses  primeiros meses últimos meses  primeiros meses últimos meses
meses                                                                                                 
1                     1093             2             68.3           0.0             18.0           0.0
2                     1115          1115             72.8          78.5             16.9          12.3
3                     1115          1115             83.2          73.7             35.4          12.3
4                     1115          1115             82.6          77.2             21.5          12.3
6                     1115          1115             36.3          80.4             14.5          16.9
9                     1115          1115             83.8          81.5             41.5          18.5
12                    1115          1115             86.7          88.0  

### 8.2 Fronteira entre clusters
Margem = quanto o segundo centroide mais próximo está mais longe que o primeiro, em proporção: (d2 − d1) / d2.
Comparo com a permanência medida no teste de estabilidade (lojas abaixo de 80% = fronteira).

In [4]:
distancias = kmeans.transform(escalonar(base))
ordenadas = np.sort(distancias, axis=1)
margem = pd.Series((ordenadas[:, 1] - ordenadas[:, 0]) / ordenadas[:, 1], index=base.index)
fronteira_real = plano.loc[base.index, 'permanencia_cluster'] < 0.80

linhas = []
for limiar in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]:
    marcada = margem < limiar
    linhas.append({'limiar_margem': limiar, 'lojas_marcadas': int(marcada.sum()),
                   'pegou_das_122_fronteira_pct': (marcada & fronteira_real).sum() / fronteira_real.sum() * 100,
                   'marcadas_que_sao_fronteira_pct': (marcada & fronteira_real).sum() / max(marcada.sum(), 1) * 100})
calibracao_margem = pd.DataFrame(linhas)
r, p = calibracao_margem['pegou_das_122_fronteira_pct'], calibracao_margem['marcadas_que_sao_fronteira_pct']
calibracao_margem['f1'] = 2 * r * p / (r + p) / 100
print(calibracao_margem.to_string(index=False, formatters={'limiar_margem': '{:.2f}'.format, 'pegou_das_122_fronteira_pct': '{:.1f}'.format,
                                                           'marcadas_que_sao_fronteira_pct': '{:.1f}'.format, 'f1': '{:.3f}'.format}))
print('\nmargem mediana - fronteira:', round(margem[fronteira_real].median(), 3), '| estáveis:', round(margem[~fronteira_real].median(), 3))

limiar_margem  lojas_marcadas pegou_das_122_fronteira_pct marcadas_que_sao_fronteira_pct    f1
         0.05              64                        43.4                           82.8 0.570
         0.10             154                        70.5                           55.8 0.623
         0.15             250                        84.4                           41.2 0.554
         0.20             348                        95.9                           33.6 0.498
         0.25             438                        98.4                           27.4 0.429
         0.30             521                        99.2                           23.2 0.376

margem mediana - fronteira: 0.055 | estáveis: 0.349


### 8.3 Perfil fora do que o modelo viu
Distância de cada loja ao centroide do próprio cluster; acima do percentil 99 do cluster no treino, a loja é marcada como atípica.

In [5]:
distancia_propria = pd.Series(distancias[np.arange(len(base)), base['cluster'].to_numpy()], index=base.index)
limites_distancia = distancia_propria.groupby(base['cluster']).quantile(0.99)
print(pd.DataFrame({'mediana': distancia_propria.groupby(base['cluster']).median(),
                    'p99': limites_distancia, 'maxima': distancia_propria.groupby(base['cluster']).max()}).round(3).T.to_string())

cluster      0      1      2      3      4
mediana  1.592  1.425  2.059  1.619  1.740
p99      3.192  2.611  6.079  3.370  3.811
maxima   4.261  4.053  6.436  5.036  4.083


### 8.4 Regras exportadas para o deploy
| Regra | Valor | Evidência |
|---|---|---|
| Histórico mínimo para classificar | 12 meses | Abaixo disso, o pior cluster acerta no máximo 41,5% (janela inicial) e 18,5% (janela final) |
| Classificação definitiva | 24 meses ou mais | Concordância de 96,6% e 96,1%; entre 12 e 23 meses fica 86,7% a 91,0%, por isso é provisória |
| Fronteira de cluster | margem < 0,10 | Maior F1 entre os limiares testados (0,623) contra as 122 lojas de baixa permanência |
| Perfil atípico | distância ao centroide > percentil 99 do cluster no treino | Limites de 2,61 (cluster 1) a 6,08 (cluster 2) |

O pacote fica em `codigo/` e usa o escalonador e o K-Means salvos, sem novo ajuste. O artefato único é `modelos/modelo_clusterizacao.joblib`.

In [2]:
print(open(RAIZ / 'modelos' / 'modelo_clusterizacao.json', encoding='utf-8').read())

{
  "versao": "1.0.0",
  "data_exportacao": "2026-10-15",
  "features": [
    "venda_media_dia",
    "ticket_medio",
    "efeito_promo",
    "volatilidade",
    "sazonalidade",
    "distancia_concorrente"
  ],
  "transformacao": "log1p nas seis features -> StandardScaler -> KMeans (k=5)",
  "mediana_distancia_concorrente": 2325.0,
  "clusters": {
    "0": {
      "nome": "Bairro competitivo de cesta pequena",
      "lojas_no_treino": 292,
      "benchmark_venda_media_dia": 5476.370334620335,
      "limite_distancia_atipica": 3.191668092195348
    },
    "1": {
      "nome": "Destino de cesta grande",
      "lojas_no_treino": 414,
      "benchmark_venda_media_dia": 6959.924351302621,
      "limite_distancia_atipica": 2.6109280301980213
    },
    "2": {
      "nome": "Sazonal",
      "lojas_no_treino": 65,
      "benchmark_venda_media_dia": 6324.801026957638,
      "limite_distancia_atipica": 6.078841253792753
    },
    "3": {
      "nome": "Dependente de promoção",
      "lojas_no_tre